# 🧹 Notebook 2 — Data Cleaning

**Project:** Customer Churn Prediction  
**Objective:** Fix all data quality issues found in Notebook 1 so the dataset is ready for EDA and modelling.

**Steps covered:**
1. Drop irrelevant columns (`customerID`)
2. Convert `TotalCharges` to numeric
3. Handle resulting NaN values
4. Strip whitespace from string columns
5. Remove exact duplicates
6. Validate cleaned dataset

---

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join('..', 'src'))
from utils import load_dataset
from preprocessing import clean_data, inspect_dataset

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')
print('Ready ✅')

## 2.1  Load Raw Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'customer_churn.csv')
df_raw = load_dataset(DATA_PATH)
print(f'Raw shape: {df_raw.shape}')

## 2.2  Investigate TotalCharges

In [ ]:
# TotalCharges is object dtype — let's see why
print(f'TotalCharges dtype: {df_raw["TotalCharges"].dtype}')

# Find rows where conversion would fail
non_numeric = df_raw[pd.to_numeric(df_raw['TotalCharges'], errors='coerce').isna()]
print(f'\nRows with non-numeric TotalCharges: {len(non_numeric)}')
if len(non_numeric) > 0:
    print(non_numeric[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

## 2.3  Apply Cleaning Pipeline

In [ ]:
# Run the clean_data function from preprocessing.py
# This encapsulates all cleaning steps in one reusable function
df_clean = clean_data(df_raw)

print(f'Raw shape  : {df_raw.shape}')
print(f'Clean shape: {df_clean.shape}')
print(f'Rows removed: {df_raw.shape[0] - df_clean.shape[0]}')

## 2.4  Validate Cleaning Results

In [ ]:
# Check for any remaining NaN values
remaining_nulls = df_clean.isnull().sum()
print('Remaining null values after cleaning:')
print(remaining_nulls[remaining_nulls > 0] if remaining_nulls.any() else '  ✅  None!')

# Confirm TotalCharges is now float
print(f'\nTotalCharges dtype: {df_clean["TotalCharges"].dtype}')

# Confirm customerID is dropped
print(f'customerID column present: {"customerID" in df_clean.columns}')

# Duplicate check
print(f'Duplicate rows remaining: {df_clean.duplicated().sum()}')

In [ ]:
# Statistics of TotalCharges after conversion
print('TotalCharges — Summary Statistics:')
print(df_clean['TotalCharges'].describe().to_string())

## 2.5  Before vs After Comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'NaN values', 'Duplicate rows', 'TotalCharges dtype'],
    'Before Cleaning': [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum(),
        str(df_raw['TotalCharges'].dtype)
    ],
    'After Cleaning': [
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum(),
        str(df_clean['TotalCharges'].dtype)
    ]
})
print(comparison.to_string(index=False))

In [ ]:
# Preview cleaned dataset
df_clean.head()

## 2.6  Save Cleaned Dataset (optional checkpoint)

In [ ]:
# Optionally save the cleaned CSV for reference
cleaned_path = os.path.join('..', 'data', 'customer_churn_cleaned.csv')
df_clean.to_csv(cleaned_path, index=False)
print(f'Cleaned dataset saved to: {cleaned_path}')

---
## Summary

| Action | Details |
|--------|---------|
| Dropped columns | `customerID` (non-predictive identifier) |
| Type conversion | `TotalCharges` → `float64` |
| Rows removed | 11 rows with whitespace TotalCharges (new customers, tenure=0) |
| Duplicates | None found |

➡  **Next:** `03_eda.ipynb` — Exploratory Data Analysis on the cleaned dataset.